# Projet Machine Learning — Prédiction du churn client

## Objectif

L’objectif de ce projet est d’analyser les données clients d’une entreprise télécom afin d’identifier les facteurs liés au churn et de construire un modèle de Machine Learning capable de prédire les clients à risque.

## 1. Import des librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import joblib

## 2. Chargement des données

In [ ]:
df = pd.read_csv("../data/customer_churn_raw.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()


In [ ]:
df.head()

## 3. Compréhension et qualité des données

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique().sort_values()

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True) * 100

In [ ]:
df["TotalCharges"].head(10)

In [ ]:
df["TotalCharges"].unique()[:20]

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df["TotalCharges"].isnull().sum()

## 4. Nettoyage des données

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

In [ ]:
df_clean[df_clean["TotalCharges"].isnull()]

In [ ]:
df_clean = df_clean.dropna(subset=["TotalCharges"])

In [ ]:
df_clean["TotalCharges"].isnull().sum()

In [ ]:
df_clean.shape

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean = df_clean.drop(columns=["customerID"])

In [ ]:
df_clean.info()

In [ ]:
df_clean.to_csv("../data/customer_churn_clean.csv", index=False)

### Synthèse du nettoyage

La colonne `TotalCharges` était initialement reconnue comme une variable texte, alors qu’elle représente un montant numérique.

Après conversion en nombre, 11 valeurs manquantes ont été détectées. Ces lignes correspondent à des clients avec une ancienneté nulle ou très récente. Comme elles représentent une très faible part du dataset, elles ont été supprimées.

La colonne `customerID`, qui correspond à un identifiant client unique, a également été retirée car elle n’apporte pas d’information utile pour la modélisation.

Après nettoyage, le dataset contient 7 032 lignes exploitables.

## 5. Analyse exploratoire des données

L’objectif de cette étape est d’analyser la répartition du churn et d’identifier les variables qui semblent influencer le départ des clients.

In [ ]:
churn_counts = df_clean["Churn"].value_counts()
churn_percent = df_clean["Churn"].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    "Nombre de clients": churn_counts,
    "Pourcentage": churn_percent.round(2)
})

churn_summary

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df_clean,
    x="Churn"
)

plt.title("Répartition des clients selon le churn")
plt.xlabel("Churn")
plt.ylabel("Nombre de clients")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
contract_churn = (
    df_clean
    .groupby("Contract")["Churn"]
    .value_counts(normalize=True)
    .rename("Proportion")
    .reset_index()
)

contract_churn["Proportion"] = contract_churn["Proportion"] * 100

contract_churn

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=contract_churn,
    x="Contract",
    y="Proportion",
    hue="Churn"
)

plt.title("Taux de churn selon le type de contrat")
plt.xlabel("Type de contrat")
plt.ylabel("Proportion (%)")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_contract.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df_clean,
    x="tenure",
    hue="Churn",
    bins=30,
    kde=True
)

plt.title("Distribution de l'ancienneté selon le churn")
plt.xlabel("Ancienneté du client en mois")
plt.ylabel("Nombre de clients")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_tenure.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_clean,
    x="Churn",
    y="MonthlyCharges"
)

plt.title("Montant mensuel selon le churn")
plt.xlabel("Churn")
plt.ylabel("Montant mensuel")

plt.tight_layout()
plt.savefig("../outputs/figures/monthly_charges_by_churn.png", dpi=300, bbox_inches="tight")
plt.show()

### Premiers constats exploratoires

L'analyse exploratoire montre que le churn concerne environ 26,5 % des clients, ce qui indique un déséquilibre modéré de la variable cible.

Le type de contrat semble fortement lié au churn : les clients avec un contrat mensuel (`Month-to-month`) présentent un taux de churn nettement plus élevé que les clients engagés sur un ou deux ans.

L'ancienneté est également un facteur important. Les clients qui quittent l'entreprise sont souvent des clients récents, ce qui montre que les premiers mois de la relation client sont particulièrement critiques.

Enfin, les clients qui churnent semblent avoir des montants mensuels plus élevés, ce qui suggère que le prix ou la perception de valeur du service peut influencer le départ des clients.

In [ ]:
internet_churn = (
    df_clean
    .groupby("InternetService")["Churn"]
    .value_counts(normalize=True)
    .rename("Proportion")
    .reset_index()
)

internet_churn["Proportion"] = internet_churn["Proportion"] * 100

internet_churn

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=internet_churn,
    x="InternetService",
    y="Proportion",
    hue="Churn"
)

plt.title("Taux de churn selon le service Internet")
plt.xlabel("Service Internet")
plt.ylabel("Proportion (%)")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_internet_service.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
payment_churn = (
    df_clean
    .groupby("PaymentMethod")["Churn"]
    .value_counts(normalize=True)
    .rename("Proportion")
    .reset_index()
)

payment_churn["Proportion"] = payment_churn["Proportion"] * 100

payment_churn

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=payment_churn,
    x="PaymentMethod",
    y="Proportion",
    hue="Churn"
)

plt.title("Taux de churn selon le moyen de paiement")
plt.xlabel("Moyen de paiement")
plt.ylabel("Proportion (%)")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_payment_method.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
online_security_churn = (
    df_clean
    .groupby("OnlineSecurity")["Churn"]
    .value_counts(normalize=True)
    .rename("Proportion")
    .reset_index()
)

online_security_churn["Proportion"] = online_security_churn["Proportion"] * 100

online_security_churn

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=online_security_churn,
    x="OnlineSecurity",
    y="Proportion",
    hue="Churn"
)

plt.title("Taux de churn selon la sécurité en ligne")
plt.xlabel("Sécurité en ligne")
plt.ylabel("Proportion (%)")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_online_security.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tech_support_churn = (
    df_clean
    .groupby("TechSupport")["Churn"]
    .value_counts(normalize=True)
    .rename("Proportion")
    .reset_index()
)

tech_support_churn["Proportion"] = tech_support_churn["Proportion"] * 100

tech_support_churn

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=tech_support_churn,
    x="TechSupport",
    y="Proportion",
    hue="Churn"
)

plt.title("Taux de churn selon le support technique")
plt.xlabel("Support technique")
plt.ylabel("Proportion (%)")

plt.tight_layout()
plt.savefig("../outputs/figures/churn_by_tech_support.png", dpi=300, bbox_inches="tight")
plt.show()

### Synthèse de l’analyse exploratoire

L’analyse exploratoire met en évidence plusieurs facteurs associés au churn.

Les clients ayant un contrat mensuel (`Month-to-month`) quittent beaucoup plus souvent l’entreprise que ceux engagés sur un ou deux ans.  
L’ancienneté du client joue également un rôle important : les clients récents sont davantage exposés au churn.

Les clients qui churnent présentent aussi des montants mensuels plus élevés en moyenne.

Concernant les services, les clients utilisant l’offre `Fiber optic` semblent quitter davantage l’entreprise que ceux utilisant `DSL`.  
Le moyen de paiement influence également le churn : les clients payant par `Electronic check` présentent un taux de churn plus élevé que ceux utilisant des paiements automatiques.

Enfin, les services additionnels comme `OnlineSecurity` et `TechSupport` semblent réduire le risque de churn. Les clients qui ne bénéficient pas de ces services quittent plus souvent l’entreprise.

Ces observations suggèrent que le churn est influencé à la fois par le niveau d’engagement contractuel, l’ancienneté du client, le coût du service et la présence de services complémentaires.

In [ ]:
df_model = df_clean.copy()

df_model["Churn"] = df_model["Churn"].map({"No": 0, "Yes": 1})

df_model["Churn"].value_counts()

In [ ]:
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

In [ ]:
X_encoded = pd.get_dummies(X, drop_first=True)

print("Dimensions après encodage :", X_encoded.shape)
X_encoded.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standardisation terminée.")

In [ ]:
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)
print("X_train_scaled shape :", X_train_scaled.shape)
print("X_test_scaled shape  :", X_test_scaled.shape)

print("\nRépartition de y_train :")
print(y_train.value_counts(normalize=True) * 100)

print("\nRépartition de y_test :")
print(y_test.value_counts(normalize=True) * 100)

## 7. Construction des modèles de Machine Learning

L’objectif est d’entraîner plusieurs modèles de classification afin de prédire si un client risque de quitter l’entreprise.

Trois modèles sont testés :
- Régression logistique ;
- Arbre de décision ;
- Random Forest.

Les modèles seront comparés à l’aide de plusieurs métriques : accuracy, precision, recall et f1-score.

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

    print(f"Résultats du modèle : {model_name}")
    print("-" * 40)
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print("\nRapport de classification :")
    print(classification_report(y_true, y_pred))

    return results

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(X_train_scaled, y_train)

y_pred_log_reg = log_reg.predict(X_test_scaled)

log_reg_results = evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_log_reg
)

In [ ]:
decision_tree = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

decision_tree.fit(X_train, y_train)

y_pred_tree = decision_tree.predict(X_test)

tree_results = evaluate_model(
    "Decision Tree",
    y_test,
    y_pred_tree
)

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=8,
    class_weight="balanced"
)

random_forest.fit(X_train, y_train)

y_pred_rf = random_forest.predict(X_test)

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    y_pred_rf
)

## 8. Évaluation et comparaison des modèles

Les modèles sont comparés à l’aide des métriques principales de classification.
Dans un contexte de churn, le recall est particulièrement important, car l’objectif est d’identifier un maximum de clients réellement à risque.

In [ ]:
model_results = pd.DataFrame([
    log_reg_results,
    tree_results,
    rf_results
])

model_results

In [ ]:
model_results_sorted = model_results.sort_values(
    by="F1-score",
    ascending=False
).reset_index(drop=True)

model_results_sorted

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=model_results_sorted,
    x="Model",
    y="F1-score"
)

plt.title("Comparaison des modèles selon le F1-score")
plt.xlabel("Modèle")
plt.ylabel("F1-score")

plt.tight_layout()
plt.savefig("../outputs/figures/model_comparison_f1_score.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
best_model_name = model_results_sorted.loc[0, "Model"]
best_model_name

In [ ]:
best_predictions = {
    "Logistic Regression": y_pred_log_reg,
    "Decision Tree": y_pred_tree,
    "Random Forest": y_pred_rf
}

y_pred_best = best_predictions[best_model_name]

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title(f"Matrice de confusion - {best_model_name}")
plt.xlabel("Prédiction")
plt.ylabel("Valeur réelle")

plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrix_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

### Interprétation des performances du meilleur modèle

Le modèle Random Forest obtient le meilleur F1-score parmi les modèles testés.

La matrice de confusion montre que le modèle identifie correctement une grande partie des clients qui churnent. Il détecte 292 clients churn sur 374 dans l’échantillon de test, soit un recall d’environ 78 % pour la classe churn.

Cependant, le modèle génère aussi des faux positifs : 269 clients sont prédits comme churn alors qu’ils ne quittent pas l’entreprise. Dans un contexte métier, cela peut être acceptable si l’objectif est de détecter un maximum de clients à risque afin de déclencher des actions de rétention.

Le modèle présente donc un bon compromis pour une première approche, mais pourrait être amélioré en ajustant le seuil de classification ou en optimisant les hyperparamètres.

## 9. Interprétation du modèle

L’objectif est d’identifier les variables les plus importantes dans la prédiction du churn afin de relier les résultats du modèle aux insights métier.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 des variables les plus importantes - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Variable")

plt.tight_layout()
plt.savefig("../outputs/figures/random_forest_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

### Interprétation des variables importantes

Le modèle Random Forest identifie `tenure`, `TotalCharges`, `Contract_Two year`, `MonthlyCharges`, `InternetService_Fiber optic` et `PaymentMethod_Electronic check` comme les variables les plus importantes dans la prédiction du churn.

Ces résultats sont cohérents avec l’analyse exploratoire : les clients récents, les clients avec des contrats mensuels, les montants mensuels élevés et certains modes de paiement présentent un risque de churn plus élevé.

À l’inverse, les contrats longue durée ainsi que certains services additionnels comme `OnlineSecurity` et `TechSupport` semblent associés à une meilleure fidélisation.

## 10. Sauvegarde du modèle final

Le modèle Random Forest est retenu comme modèle final car il obtient le meilleur F1-score parmi les modèles testés.

In [ ]:
final_model = random_forest

model_artifact = {
    "model": final_model,
    "features": X_train.columns.tolist()
}

joblib.dump(model_artifact, "../models/churn_random_forest_model.pkl")

In [ ]:
import os

os.listdir("../models")

## 11. Recommandations business

### Recommandations business

À partir de l’analyse exploratoire et des résultats du modèle, plusieurs recommandations peuvent être proposées afin de réduire le churn client.

1. **Renforcer l’accompagnement des nouveaux clients**

L’ancienneté (`tenure`) est la variable la plus importante dans le modèle. Les clients récents présentent un risque de churn plus élevé.  
Il serait donc pertinent de mettre en place un parcours d’onboarding renforcé pendant les premiers mois : suivi personnalisé, messages de bienvenue, assistance proactive et offres de fidélisation.

2. **Encourager les contrats longue durée**

Les clients avec des contrats `One year` ou `Two year` churnent beaucoup moins que ceux avec des contrats mensuels.  
L’entreprise pourrait proposer des avantages spécifiques pour inciter les clients à passer d’un contrat mensuel à un contrat annuel ou biannuel.

3. **Surveiller les clients avec des charges mensuelles élevées**

Les clients qui churnent ont tendance à avoir des `MonthlyCharges` plus élevés.  
Une analyse complémentaire pourrait être menée pour identifier les clients ayant un coût élevé mais une faible utilisation des services, afin de leur proposer des offres mieux adaptées.

4. **Analyser l’offre Fiber optic**

Les clients utilisant `Fiber optic` présentent un taux de churn plus élevé.  
Cela peut indiquer un problème de prix, de qualité perçue, de concurrence ou de satisfaction client. Une enquête ciblée sur cette population pourrait permettre d’identifier les causes précises.

5. **Réduire le risque lié au paiement par Electronic check**

Le moyen de paiement `Electronic check` est associé à un churn plus élevé.  
L’entreprise pourrait encourager les paiements automatiques par carte bancaire ou virement, qui semblent associés à une meilleure fidélisation.

6. **Promouvoir les services additionnels**

Les clients disposant de services comme `OnlineSecurity` et `TechSupport` churnent moins.  
Ces services peuvent être utilisés comme leviers de rétention, notamment pour les clients à risque.

7. **Mettre en place un score de risque de churn**

Le modèle peut être utilisé pour attribuer un score de risque à chaque client.  
Les équipes marketing ou relation client pourraient ensuite prioriser les actions de rétention sur les clients les plus à risque.

## 12. Conclusion

Ce projet a permis d’analyser les facteurs associés au churn client et de construire un modèle de Machine Learning capable d’identifier les clients à risque.

L’analyse exploratoire a montré que le churn est fortement lié à l’ancienneté du client, au type de contrat, au montant mensuel, au service Internet, au moyen de paiement et à la présence de services additionnels.

Après comparaison de plusieurs modèles, le Random Forest a été retenu comme modèle final grâce à son meilleur F1-score. Le modèle présente un bon recall sur la classe churn, ce qui est important dans un contexte de rétention client.

Ce projet montre que l’analyse de données et le Machine Learning peuvent aider une entreprise à mieux comprendre les causes du churn et à cibler plus efficacement ses actions de fidélisation.

## 13. Export des résultats

In [ ]:
# Export du dataset nettoyé
df_clean.to_csv("../data/customer_churn_clean.csv", index=False)

# Export des résultats des modèles
model_results_sorted.to_csv("../outputs/results/model_comparison.csv", index=False)

# Export de l'importance des variables
feature_importance.to_csv("../outputs/results/feature_importance.csv", index=False)

print("Exports terminés.")

In [ ]:
import os

print("Fichiers dans outputs/results :")
print(os.listdir("../outputs/results"))

print("\nFichiers dans models :")
print(os.listdir("../models"))

## 14. Synthèse finale du projet

Ce projet avait pour objectif d’analyser le churn client d’une entreprise télécom et de construire un modèle de Machine Learning capable d’identifier les clients à risque.

L’analyse exploratoire a montré que les facteurs les plus associés au churn sont l’ancienneté du client, le type de contrat, le montant mensuel, le service Internet, le moyen de paiement, la sécurité en ligne et le support technique.

Les clients les plus à risque sont principalement des clients récents, avec un contrat mensuel, des charges mensuelles élevées, un paiement par Electronic check, sans sécurité en ligne et sans support technique.

Plusieurs modèles ont été testés : régression logistique, arbre de décision et Random Forest. Le modèle Random Forest a obtenu le meilleur F1-score et a été retenu comme modèle final.

Ce projet montre comment l’analyse de données et le Machine Learning peuvent aider une entreprise à mieux cibler ses actions de fidélisation et à réduire le churn.